# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and preparing the FAIR^2 dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library in Python.

### Dataset Source
The dataset is described by a Croissant schema and can be accessed via the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display core metadata
print("Dataset name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Version:", getattr(dataset.metadata, 'version', None))

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We will:
- List each record set's `@id` and name.
- For each, list available fields and their `@id` values.
This helps us reference entities by their unique `@id`s, following best practice for the Croissant schema.

In [ ]:
# Examine available record sets
record_sets = list(dataset.record_sets)
print("Available record sets (@id and name):\n")
for rs in record_sets:
    print(f"  @id: {rs.id}    name: {getattr(rs, 'name', None)}")

# For each record set, list available fields and their @id values
print('\nFields per record set:')
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}")
    for field in rs.fields:
        print(f"    Field @id: {field.id}     name: {getattr(field, 'name', None)}")

## 3. Data Extraction
Now we'll select one or more record sets (by `@id`) and load their records using `mlcroissant`, referencing all entities by their `@id`. We'll collect the records as pandas DataFrames for further analysis.

> _**Note:** The field and column references below use their `@id` values as discovered in the overview above!_


In [ ]:
# Define the record set(s) you want to extract, using the `@id`s from above.
# For this dataset, let's extract ALL available record sets.
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} (columns: {dataframes[record_set_id].columns.tolist()})\n")
    else:
        print(f"No records found for {record_set_id}.\n")
if len(dataframes) == 0:
    print("No record sets contained tabular data.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field from a record set and perform some basic exploratory operations: filtering, normalization, and grouping.

- _Edit the `numeric_field_id` and `group_field_id` below to match one that exists in your extracted DataFrame columns (use the `@id` value as column name if possible, or actual field names)._

In [ ]:
# For demonstration, choose the first available DataFrame
if dataframes:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}\nColumns:\n{df.columns}\n")
    # Try to automatically pick a likely numeric field (e.g., the first float/integer-typed column)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found in DataFrame; EDA will be skipped.")
    else:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())
        # Try to pick a categorical field to group by
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == "object":
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (showing means):")
            print(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field, and a boxplot grouped by the chosen categorical field if available.

> _If you see an empty plot or error, check that the right numeric and group fields are being chosen in the EDA cell above._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'df' in locals() and numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant-structured dataset with `mlcroissant`, examine its record sets and fields using their `@id` values, extract records to pandas DataFrames, and carry out basic exploratory analysis and visualizations. For further analysis, consult the dataset's metadata for domain-specific field meanings, filtering, and custom pre-processing steps.

You can now extend this notebook to perform deeper domain-specific analyses or to prepare the data for machine learning tasks!